# Phase 12 — Deployment

This notebook records reproducible evidence for the public FinAccess Eswatini deployment. It validates the selected Signal frontend, the same-origin FastAPI service, both model/explainer pairs, input validation, prediction equivalence, and deployment-repository safety.

It does not begin Phase 13 portfolio polish.

In [1]:
from finaccess_eswatini.phase12_deployment import run

deployment = run()
print(f"Phase status: {deployment['status'].replace('_', ' ')}")
print(f"Validated at: {deployment['generated_at_utc']}")
print(f"Frontend: {deployment['endpoints']['frontend']}")
print(f"API: {deployment['endpoints']['api']}")

Phase status: PASS WITH NOTES
Validated at: 2026-08-11T13:40:33.027284+00:00
Frontend: https://finaccess-eswatini.vercel.app
API: https://finaccess-eswatini.vercel.app/api


## Public validation checks

In [2]:
import pandas as pd

checks = pd.DataFrame(deployment['checks'])
checks

,check,status,evidence
0,public_frontend_https,PASS,HTTP 200 at the production domain; Signal head...
1,api_health_and_artifact_integrity,PASS,HTTP 200; two hash-matched model/explainer pai...
2,openapi_contract,PASS,HTTP 200; same-origin health and assessment pa...
3,interactive_api_documentation,PASS,HTTP 200 at /api/docs.
4,combined_assessment,PASS,HTTP 200; one public request returned both out...
5,validated_prediction_equivalence,PASS,Live probabilities and all local SHAP factors ...
6,explanation_factors,PASS,Model-derived explanation counts: {'financial_...
7,invalid_input_rejected,PASS,Empty profile rejected with HTTP 422.
8,one_domain_no_cors_dependency,PASS,The browser and FastAPI routes share one HTTPS...
9,vercel_services_configuration,PASS,Next.js and FastAPI are separate Vercel servic...


## End-to-end smoke-test output

In [3]:
for outcome, result in deployment['sample_prediction'].items():
    label = outcome.replace('_', ' ').title()
    print(f"{label}: {result['answer']}")
    print(f"  Estimated likelihood: {result['probability_percent']}%")
    print(f"  Model-generated factors: {result['factor_count']}")

Financial Inclusion: This person is unlikely to be financially included.
  Estimated likelihood: 26.9%
  Model-generated factors: 5
Mobile Money Adoption: This person is unlikely to use mobile money.
  Estimated likelihood: 36.8%
  Model-generated factors: 5


## Point-in-time response observations

In [4]:
pd.Series(deployment['latency_seconds'], name='seconds').to_frame()

,seconds
frontend,1.673
health,9.684
api_docs,0.442
assessment,0.729
invalid_request,0.640


## Deployment decisions and limitations

- One Vercel Hobby project deploys the standard Next.js frontend and FastAPI backend as separate Vercel Services.
- The frontend posts directly to `/api/v1/assessment` on the same public domain, so no cross-host proxy, external API URL, or browser CORS configuration is required.
- The FastAPI service verifies the SHA-256 digest of each pipeline and model-matched SHAP explainer before reporting healthy.
- Raw and processed respondent microdata are excluded from the deployment repository.
- Vercel Services and its Python runtime are beta features; the optimized Python bundle size and serverless cold-start behaviour remain deployment risks to monitor.
- Automatic Git deployments require granting the Vercel GitHub App access to the private web repository; this validated release used the authenticated Vercel CLI.
- The system remains a portfolio proof of concept, not a production financial decision engine.